In [1]:
import argparse
from scapy.all import rdpcap, wrpcap, IP, TCP, UDP, ICMP  # noqa: F401

def parse_mapping(mapping_str):
    """
    Converte una stringa tipo '1.1.1.1=10.0.0.1,2.2.2.2=10.0.0.2'
    in un dict {'1.1.1.1': '10.0.0.1', ...}
    """
    if not mapping_str:
        return {}
    out = {}
    for pair in mapping_str.split(","):
        old, new = pair.split("=")
        out[old.strip()] = new.strip()
    return out

def rewrite_pcap(input_pcap, output_pcap, ip_map, time_shift, time_scale):
    pkts = rdpcap(input_pcap)
    if not pkts:
        raise RuntimeError("Il pcap è vuoto!")

    first_time = pkts[0].time
    for pkt in pkts:
        # --- IP rewriting ---
        if IP in pkt:
            ip_layer = pkt[IP]
            if ip_layer.src in ip_map:
                ip_layer.src = ip_map[ip_layer.src]
            if ip_layer.dst in ip_map:
                ip_layer.dst = ip_map[ip_layer.dst]

            # Reset checksums – Scapy li ricalcola se li elimini
            if TCP in pkt:
                del pkt[TCP].chksum
            if UDP in pkt:
                del pkt[UDP].chksum
            del ip_layer.chksum

        # --- timing rewrite ---
        rel_t = pkt.time - first_time              # tempo relativo (s)
        new_rel = rel_t * time_scale + time_shift  # scala + shift
        pkt.time = first_time + new_rel            # nuovo timestamp assoluto

    wrpcap(output_pcap, pkts)
    print(f"Creato '{output_pcap}' con {len(pkts)} pacchetti")

def main():
    ip_map = parse_mapping("192.168.1.1=10.0.0.1, 192.168.1.13=10.0.0.18,192.168.1.19=10.0.0.17, 192.168.1.21=10.0.0.12, 192.168.1.22=10.0.0.13, 192.168.1.24=10.0.0.14,192.168.1.25=10.0.0.15,192.168.1.26=10.0.0.16,192.168.1.27=10.0.0.99, 192.168.1.28=10.0.0.99, 192.168.1.29=10.0.0.99, 192.168.1.255=10.0.0.255")
    rewrite_pcap("20200426_UOWM_IEC104_Dataset_c_ci_na_1_DoS/20200426_UOWM_IEC104_Dataset_c_ci_na_1_DoS_attacker1/20200426_UOWM_IEC104_Dataset_c_ci_na_1_DoS_attacker1_iec104_only.pcap", "c_ci_na_parsed.pcap", ip_map, 0, 1)

if __name__ == "__main__":
    main()


Creato 'c_ci_na_parsed.pcap' con 2396 pacchetti


In [2]:
pip install scapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
